# **Feature Exploration**

## Objectives

* Investigate whether domain-informed derived features improve the predictive power of the data

## Inputs

* "outputs/datasets/cleaned/HotelBookingsClean.csv"
* "outputs/correlation/RankedFeatures.csv" 

## Outputs

* outputs/exploration/TopFeatures.csv

## Additional Comments

* This notebook extends the [correlation study](/jupyter_notebooks/05_correlation_study.ipynb) to explore *derived* features prior to formal feature engineering
* Candidate transformatins were selected using hotel revenue management knowledge as well as statstical observations
* Every derived feature was evaluated using the same methodology as notebook 5 to maintain consistency
* Features are only retained if the improve predictive signal or offer a meaningful simplification for modelling

---

# Change working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [ ]:
import os
current_dir = os.getcwd()
current_dir

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chdir() defines the new current directory

In [ ]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

current_dir = os.getcwd()
current_dir

## Load Data

* Load cleaned dataset

In [ ]:
import pandas as pd
df = pd.read_csv("outputs/datasets/cleaned/HotelBookingsClean.csv")
df.head(3)


In [ ]:
df.info()

* Re-run categorical type changes

In [ ]:
categorical_list = ["is_repeated_guest", "agent", "company"]
for col in categorical_list:
    df[col] = df[col].astype("category")

* Load ranked features

In [ ]:
ranked_features = pd.read_csv("outputs/correlation/RankedFeatures.csv")
ranked_features.sort_values(by=["feature_type", "rank_in_type"])

---

## Investigate domain-informed, derived features

### Strategy

| Candidate(s) | Reason(s) | Transformation(s) |
| --- | --- | --- |
| adults, children, babies | Occupancy represented by 3 variables, family-type booking not well represented | total_guests unified feature, is_family binary flag |
| stays_in_week_nights, stays_in_weekend_nights | Length of stay represented by 2 variables, midweek/weekend not well represented | los unified feature, add is_weekend binary flag |
| arrival_date_year, arrival_date_month, arrival_date_week_number, arrival_date_day_of_month | Arrival date represented by 4 variables, cyclical calendar features not correctly represented | Drop arrival_date_year, sin/cos encoding on arrival_date_month and arrival_date_week_number, season time-bucketed column if seasonality is a predictor |
| lead_time | Current distribution heavily skewed | Group into time-buckets as explored in eda |
| previous_cancellations, previous_bookings_not_canceled | Customer history represented by 2 continuous numeric variables | Unify into has_cancelled_before binary flag |
| required_car_parking_spaces, total_of_additional_requests | Additional needs represented by 2 continuous numeric variables | Unify into has_additional_needs binary flag |
| country, agent, company | Long-tail categorical variables | Group into top_n + other |
| days_in_waiting_list | Zero-inflated, heavily skewed continuous numeric variable | Replace with was_waitlisted binary flag |

* Generate results variable and validaton function

In [ ]:
results = []

In [ ]:
# Carry the threshold values from 05_correlation_study as a starting point
threshold = 0.1
pps_threshold = 0

In [ ]:
import ppscore as pps
from pandas.api.types import is_numeric_dtype

def feature_comparison(feature, target, raw):    
    
    def is_raw():
        if feature.name in raw.columns:
            return "Raw"
        else:
            return "Engineered"

    if is_numeric_dtype(feature):    
        def pearson():
            return feature.corr(target, method="pearson")

        pearson_score = pearson()

        def spearman():
            return feature.corr(target, method="spearman")

        spearman_score = spearman()
    else:
        pearson_score = "n/a"
        spearman_score = "n/a"
    
    def pps_score():
        pps_df = raw.copy()
        pps_df[target.name] = pps_df[target.name].astype("category")
        pps_df[feature.name] = feature
        return pps.score(pps_df, x=feature.name, y=target.name)["ppscore"]
    
    pps_result = pps_score()
    
    def decision():
        pearson_yes = pearson_score != "n/a" and abs(pearson_score) > threshold
        spearman_yes = spearman_score != "n/a" and abs(spearman_score) > threshold
        
        if pearson_yes or spearman_yes or pps_result > pps_threshold:
            return "Yes"
        else:
            return "No"
        
    return {"Feature": feature.name,
            "Raw/Engineered": is_raw(),
            "Pearson": pearson_score,
            "Spearman": spearman_score,
            "PPS": pps_result,
            "Above Threshold": decision()}

* Populate `results` with raw features

In [ ]:
target = df["is_canceled"]

for col in df.columns:
    if col != "is_canceled":
        results.append(feature_comparison(df[col], target, df))

results

**Guest Composition**

* Assess whether an is_family binary flag is of more predictive value than guest counts alone

In [ ]:
is_family = pd.Series(df[["children", "babies"]].gt(0).any(axis=1), name="is_family").astype("category")
is_family.head()

In [ ]:
is_family_val = feature_comparison(is_family, target, df)
is_family_val

In [ ]:
total_guests = pd.Series(df["adults"] + df["children"] + df["babies"], name="total_guests")
total_guests.head()

* Investigate whether combining `adults`, `children` and `babies` into a total guest count provides a stronger predictive signal

In [ ]:
total_guests_val = feature_comparison(total_guests, target, df)
total_guests_val

* Add results to results list

In [ ]:
results.append(is_family_val)
results.append(total_guests_val)

* Neither engineered feature improved the predictive signal relative to the original variables. The original guest compostition features will be retained for modelling

**Stay Characteristics**

* Evaluate whether combining `stays_in_week_nights` and `stays_in_weekend_nights` into a total length of stay (LOS) improves predictive performance

In [ ]:
los = pd.Series(df["stays_in_week_nights"] + df["stays_in_weekend_nights"], name="LOS")
los.head()

In [ ]:
los_val = feature_comparison(los, target, df)
los_val

* Add to results list

In [ ]:
results.append(los_val)

* Combining weekday and weekend stays into a single length-of-stay feature did not improve predictive performance. The original separated values are retained

**Temporal Variables**

* Use cyclical encoding on `arrival_date_month` and `arrival_date_year` to explore the effect of seasonality

In [ ]:
month_map = {"January": 1, "February": 2, "March": 3, "April": 4,
             "May": 5, "June": 6, "July": 7, "August": 8,
             "September": 9, "October": 10, "November": 11, "December": 12}
month = df["arrival_date_month"].map(month_map)
month.head()



In [ ]:
from feature_engine.creation import CyclicalFeatures
import numpy as np

month = np.array(month).reshape(-1, 1)
cyclical = CyclicalFeatures(variables=None, drop_original=False)
month_cycle = cyclical.fit_transform(month)
month_cycle.head()

---

In [ ]:
month_test = pd.concat([target, month_cycle], axis=1)
month_test.head()

In [ ]:
month_test["is_canceled"] = month_test["is_canceled"].astype("category")
month_pps = pps.predictors(month_test, y="is_canceled")
month_pps

* Cyclical encoding of `arrival_date_month` did not improve predictive performance, indicating that seasonal effects are not a major driver of booking cancellation within this dataset
* `arrival_date_year` will be excluded from modelling to avoid introducing dataset-specific temporal patterns

**Booking Behaviour**

* Explore whether lead time benefits from transformation or categorisation

In [ ]:
bins = [-np.inf, 7 ,30 ,90, np.inf]
lead_time_cat = pd.cut(df["lead_time"], bins, labels=["Last Minute", "Short Range", "Mid Range", "Long Range"])
lead_time_cat.name = "lead_time_category"
lead_time_cat.head()

In [ ]:
lead_time_val = feature_comparison(lead_time_cat, target, df)
lead_time_val

* Categorising `lead_time` into booking windows produced a modest improvement in PPS: 0.239 compared to 0.216, suggesting that cancellation behaviour is better represented by booking horizon rather than raw lead time alone. This feature will be evaluated during modelling

* Add to results list

In [ ]:
results.append(lead_time_val)

* Consider whether previous history can be better represented through derived behavioural features

In [ ]:
has_canceled_before = pd.Series(df["previous_cancellations"].gt(0)).astype("category")
has_canceled_before.name = "has_canceled_before"
has_canceled_before.head()

In [ ]:
has_canceled_before_val = feature_comparison(has_canceled_before, target, df)
has_canceled_before_val

* The binary representation caused almost no improvement in PPS. Both representations will be considered during modelling

* Add to results

In [ ]:
results.append(has_canceled_before_val)

**Long-tail compression**

* From the RankedFeatures table we can see that `country` and `agent` already have predictive signal, now we will consider if compiling them into top_n + other improves these results 

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def plot_categoric_features(cols):
    ncols = 1
    nrows = 2
    fig, axs = plt.subplots(nrows, ncols, figsize=(15, 18))
    axs = axs.flatten()

    for i, col in enumerate(cols):
        order = df[col].value_counts().index

        sns.countplot(data=df,
                     x=col,
                     order=order,
                     ax=axs[i])
        
        axs[i].set_title(df[col].name)
    
    plt.tight_layout()
    plt.show()

cols = ["country", "agent"]

plot_categoric_features(cols)


* Visually it appears as though for `country` and `agent`, a good starting point for the grouping would be n = 10

In [ ]:
# Code repeated from notebook 02_cancellation_eda
n=10
country = df["country"].value_counts(sort=True, ascending=False)
country_top_n = country.iloc[:n]
other = pd.Series(country.iloc[n:].sum(), index=["Other"])
country = pd.concat([country_top_n, other])

country = pd.DataFrame(country, columns=["count"])
country_list = country.index.unique().to_list()[:-1]


In [ ]:
country = df["country"]
mask = df["country"].isin(country_list)
country = country.where(mask, "Other")
country.name = "country_fe"
country.value_counts()

In [ ]:
country_val = feature_comparison(country, target, df)
country_val

* Country cardinality reduction was investigated due to the high number of unique values (177 categories). Grouping countries into top_n + other produced negligible changes in predictive performance, with the highest observed PPS improvement of only 0.001 from the top-10 grouping. As the transformation introduced additional assumptions and potential loss of geographic information without a meaningful predictive benefit, the original country feature was retained for model evaluation

In [ ]:
n=320
agent = df["agent"].value_counts(sort=True, ascending=False)
agent_top_n = agent.iloc[:n]
other = pd.Series(agent.iloc[n:].sum(), index=["Other"])
agent = pd.concat([agent_top_n, other])

agent = pd.DataFrame(agent, columns=["count"])
agent_list = agent.index.unique().to_list()[:-1]

In [ ]:
agent_fe = df["agent"].astype("object")
agent_fe.name = "agent_fe"
mask = df["agent"].isin(agent_list)
agent_fe = agent_fe.where(mask, "Other")

In [ ]:
df["agent"].nunique()

In [ ]:
agent_val = feature_comparison(agent_fe, target, df)
agent_val

* This transformation adds no predictive value. This suggests that agent identifiers contain meaningful predictive information and should not be grouped.
* Several thresholds for grouping infrequent IDs were evaluated. The original n value of 10 was far too low, the feature has 344 unique features so n_values of 100, 200 and 300 were tested. n=320 also returned pps=0, and since this is close to the number of unique category IDs, the original feature will be retained. 

* Add to results

In [ ]:
results.append(country_val)
results.append(agent_val)

**Waitlist**

* `days_in_waiting_list` is a long-tail continuous numeric that may benefit from a binary `was_waitlisted` flag

In [ ]:
waitlist = pd.Series(df["days_in_waiting_list"].gt(0)).astype("category")
waitlist.name = "was_waitlisted"
waitlist.head()

In [ ]:
waitlist_val = feature_comparison(waitlist, target, df)
waitlist_val

* The binary transformation did not improve predictive performance, the original feature will be retained.

* Add to results

In [ ]:
results.append(waitlist_val)

In [ ]:
results_df = pd.DataFrame(results)
results_features = results_df[results_df["Above Threshold"] == "Yes"]
results_features.sort_values(by="PPS", ascending=False)

---

## Summary Table

| Candidate | Outcome | Decision |
| --- | --- | --- |
| Total Guests | No improvement | Reject |
| Family Flag | No improvement | Reject |
| LOS | No improvement | Reject |
| Lead Time Category | Moderate improvement | Keep for model performance experimentation |
| Country FE | Similar PPS, lower cardinality, information loss | Keep for model performance experimentation |
| Has Cancelled Before | Similar PPS | Keep for model performance experimentation |
| Agent FE | Information loss | Reject |
| Waitlist Flag | No improvement | Reject |


---

## Conclusions

* Most domain-informed feature transformations made little or no improve to predictive signal.
* The original variables capture the booking behaviour effectively.
* Feature transformations are therefore applied selectively rather than indiscriminately.
* Only transformations that demonstrated measurable benefit or simplified preprocessing were retained.

## Next Steps

* Finalise preprocessing pipeline.
* Encode categorical variables.
* Train and compare classification models using the validated feature set.
* Evaluate feature importance after modelling.

# Push files to Repo

In [ ]:
import os
try:
  os.makedirs(name='outputs/correlation')
except Exception as e:
  print(e)


In [ ]:
results_features.to_csv("outputs/correlation/TopFeatures.csv")